# Milestone 3 — Static batching

The per-step floor is paid per **step**, not per token. Batching splits it.

In [1]:
!git clone https://github.com/Jayaprakash-030/tiny-inference-engine.git
%cd tiny-inference-engine
!pip install -q -e .

Cloning into 'tiny-inference-engine'...
remote: Enumerating objects: 35, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 35 (delta 9), reused 34 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (35/35), 74.09 KiB | 6.73 MiB/s, done.
Resolving deltas: 100% (9/9), done.
/content/tiny-inference-engine
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for tiny-inference-engine (pyproject.toml) ... done


In [2]:
from engine import load
from engine.batched import (
    batched_generate,
    check_batch_matches_single,
    print_generations,
    staggered_limits,
    sweep,
)
from engine.prompts import BATCH
from engine.results import save

rt = load()
rt.describe()


ImportError: cannot import name 'staggered_limits' from 'engine.batched' (/content/tiny-inference-engine/engine/batched.py)

### Correctness first

In [3]:
assert check_batch_matches_single(rt, BATCH[:4]), "batching diverged — check padding / position_ids"

batch matches single: True


In [4]:
batched_generate(rt, BATCH[:2], max_new_tokens=16)   # warmup
print("warmup done")

warmup done


### Per-seq limits (so waste is visible)

Open prompts almost never hit EOS under greedy decoding, so a shared
`max_new_tokens` makes every sequence run the same length and
`wasted_slot_pct` stays 0. Real serving gives each request its own
`max_tokens`. Staggered limits recreate that finish-time skew.


In [ ]:
MAX_NEW = 200
prompts = BATCH[:4]
limits = staggered_limits(len(prompts), MAX_NEW)
print("limits:", limits)

out, step_ms, kept, computed = batched_generate(
    rt, prompts, max_new_tokens=MAX_NEW, max_new_tokens_per_seq=limits,
)
print_generations(rt, prompts, out, MAX_NEW, limits=limits)
print(f"kept={kept} computed={computed} "
      f"wasted_slot_pct={100 * (computed - kept) / computed:.1f}%")


### Throughput sweep

`sweep(..., stagger=True)` (default) uses staggered per-seq limits so
`wasted_slot_pct` is the argument for milestone 4.


In [ ]:
m3 = sweep(rt, sizes=(1, 2, 4, 8, 16, 32), max_new_tokens=200, stagger=True)
save(m3)


In [ ]:
import matplotlib.pyplot as plt

from pathlib import Path

Path("benchmarks/plots").mkdir(parents=True, exist_ok=True)

sizes = [r["batch_size"] for r in m3]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, key, title, color in [
    (axes[0], "tokens_per_sec", "Throughput (total tok/s)", "tab:blue"),
    (axes[1], "per_seq_tok_per_sec", "Per-sequence speed (tok/s)", "tab:orange"),
    (axes[2], "peak_mem_gb", "Peak memory (GB)", "tab:green"),
]:
    ax.plot(sizes, [r[key] for r in m3], marker="o", color=color)
    ax.set_title(title)
    ax.set_xlabel("batch size")
    ax.set_xscale("log", base=2)
    ax.set_xticks(sizes)
    ax.set_xticklabels(sizes)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("benchmarks/plots/m3_batching.png", dpi=140)
plt.show()

Throughput climbs then flattens — the knee is where the GPU stops being launch-bound.
Per-sequence speed drifts *down*: throughput up, individual latency down, the central
trade in serving.

`wasted_slot_pct` is the argument for milestone 4.